### PROYECTO DE CÁLCULO
Mateo Serrato Ascencio


# Objetivo del Proyecto

El objetivo principal de este proyecto es aplicar conceptos fundamentales de cálculo vectorial para modelar, simular y visualizar un problema de optimización en un campo escalar.

Específicamente, se busca determinar la trayectoria de descenso más pronunciado para una partícula sobre la superficie definida por la función $f(x,y) = x^2 \cos(y) - y^2 \sin(x)$, desde un punto inicial aleatorio hasta alcanzar el borde de la región de estudio.

---
## Metas Específicas

Para cumplir con el objetivo general, se plantearon las siguientes metas:

1.  **Modelar la Trayectoria:** Implementar en Python un algoritmo de **descenso de gradiente** para encontrar numéricamente la secuencia de puntos que conforman la trayectoria óptima de la partícula.

2.  **Visualizar los Resultados:** Graficar el campo escalar como una superficie 3D y superponer la trayectoria calculada. Adicionalmente, generar una **animación interactiva** del recorrido de la partícula utilizando la librería Plotly para una mejor comprensión del movimiento.

3.  **Analizar el Recorrido:** Calcular la **integral de línea** del campo escalar a lo largo de la trayectoria encontrada y proporcionar una interpretación física y geométrica del valor numérico obtenido.

In [20]:
#IMPORTAMOS LIBRERIAS A UTILIZAR
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
import time
import plotly.graph_objects as go

#DEFINIMOS FUNCIONES
def f(x,y): # Campo escalar en R2 dado por: 
    #return x**2*(np.cos(y)) - (y**2)*np.sin(x)
    return np.exp(-(x**2 + y**2)/4)
def punto_inicial(): # Función que define un punto inicial de manera aleatoria 
    np.random.seed(int(time.time()))
    x_inicial = np.random.uniform(-5,5)
    y_inicial = np.random.uniform(-5,5)
    return np.array([x_inicial, y_inicial])

def gradiente(f,x,y):# Función que calcula el gradiente usando la fórmula de derivada tradicional
    h = 1e-5
    df_dx = (f(x + h, y) - f(x - h, y)) / (2*h)
    df_dy = (f(x, y + h) - f(x, y - h)) / (2*h)
    return np.array([df_dx,df_dy])

def descenso_gradiente(f, gradiente, region, punto_inicial, min_or_max, alpha=0.01, max_iter=1000, epsilon=1e-6, ruido=0.1):
    x, y = punto_inicial
    trayectoria = [(x, y)]
    
    for _ in range(max_iter):
        # Calculamos gradiente
        g = gradiente(f, x, y)
        gx, gy = g

        # Si el gradiente es muy pequeño → agregamos un empujón aleatorio
        if np.linalg.norm(g) < epsilon:
            # ruido tanto en gradiente como en posición
            gx += np.random.uniform(-ruido, ruido)
            gy += np.random.uniform(-ruido, ruido)
            x += np.random.uniform(-ruido, ruido)
            y += np.random.uniform(-ruido, ruido)

        # Actualizamos la posición
        if min_or_max == 'min':
            x_nuevo = x - alpha * gx
            y_nuevo = y - alpha * gy
        elif min_or_max == 'max':
            x_nuevo = x + alpha * gx
            y_nuevo = y + alpha * gy
        else:
            break
            
        # Verificamos que siga en la región
        if not (region[0] <= x_nuevo <= region[1] and region[0] <= y_nuevo <= region[1]):
            break
            
        x, y = x_nuevo, y_nuevo
        trayectoria.append((x, y))
        
    return np.array(trayectoria)

import numpy as np

def integral_linea(f, trayectoria):
    """
    Calcula la integral de línea de una función escalar f(x,y) sobre una trayectoria,
    aproximando la integral por la suma de productos f(x_i,y_i) * ||r_{i+1} - r_i||.

    Args:
        f (callable): función f(x, y) que acepta dos arrays o dos escalares.
        trayectoria (np.ndarray): array de forma (n,2) con los puntos (x,y) en orden.

    Returns:
        float: aproximación de la integral de línea ∫_C f(x,y) ds.
    """
    integral = 0.0
    for i in range(len(trayectoria) - 1):
        xi, yi = trayectoria[i]
        xj, yj = trayectoria[i+1]

        # valor de f en el punto inicial del segmento
        val_f = f(xi, yi)

        # longitud del segmento (multiplicador)
        dr = np.array([xj - xi, yj - yi])
        longitud = np.linalg.norm(dr)

        # multiplicación por segmento y acumulación
        integral += val_f * longitud

    return integral

In [21]:

# Parámetros para la visualización y el cálculo
region = [-5, 5]
min_or_max = 'min'  # cambiar a 'max' para encontrar un máximo

# 1. Ejecutamos el descenso de gradiente
#punto_inicio = punto_inicial()
punto_inicio = np.array([0.0,0.0])
trayectoria_encontrada = descenso_gradiente(f, gradiente, region, punto_inicio, min_or_max)

# 2. Calculamos la integral de línea a lo largo de la trayectoria
valor_integral = integral_linea(f, trayectoria_encontrada)

# 3. Preparamos los datos para la visualización
x_surface = np.linspace(region[0], region[1], 100)
y_surface = np.linspace(region[0], region[1], 100)
X, Y = np.meshgrid(x_surface, y_surface)
Z = f(X, Y)

# Puntos de la trayectoria
x_trayectoria = trayectoria_encontrada[:, 0]
y_trayectoria = trayectoria_encontrada[:, 1]
z_trayectoria = f(x_trayectoria, y_trayectoria)

# Imprimimos los resultados
print(f"Punto inicial: ({punto_inicio[0]:.2f}, {punto_inicio[1]:.2f})")
print(f"Punto final de la trayectoria: ({x_trayectoria[-1]:.2f}, {y_trayectoria[-1]:.2f})")
print(f"El valor de la integral de línea es: {valor_integral:.4f}")


# 4. Creamos la visualización 
fig = go.Figure()

# Añadimos la superficie del campo escalar
fig.add_trace(go.Surface(
    x=x_surface, y=y_surface, z=Z,
    colorscale='Viridis',
    showscale=False,
    opacity=0.8
))

# Añadimos la trayectoria completa como una línea estática
fig.add_trace(go.Scatter3d(
    x=x_trayectoria, y=y_trayectoria, z=z_trayectoria,
    mode='lines',
    line=dict(color='red', width=4),
    name='Trayectoria'
))

# Añadimos el punto inicial
fig.add_trace(go.Scatter3d(
    x=[x_trayectoria[0]], y=[y_trayectoria[0]], z=[z_trayectoria[0]],
    mode='markers',
    marker=dict(color='white', size=5, symbol='circle'),
    name='Punto Inicial'
))

# Añadimos el punto final
fig.add_trace(go.Scatter3d(
    x=[x_trayectoria[-1]], y=[y_trayectoria[-1]], z=[z_trayectoria[-1]],
    mode='markers',
    marker=dict(color='cyan', size=5, symbol='x'),
    name='Punto Final'
))

# Añadimos la partícula que se va a animar (inicialmente en el punto de partida)
fig.add_trace(go.Scatter3d(
    x=[x_trayectoria[0]], y=[y_trayectoria[0]], z=[z_trayectoria[0]],
    mode='markers',
    marker=dict(color='magenta', size=8),
    name='Partícula'
))

# Creamos los frames para la animación
frames = [go.Frame(data=[go.Scatter3d(
                    x=[x_trayectoria[k]],
                    y=[y_trayectoria[k]],
                    z=[z_trayectoria[k]])],
                   # El '4' es el índice del trace de la partícula en fig.data
                   traces=[4],
                   name=f'frame{k}') for k in range(len(x_trayectoria))]

fig.frames = frames

# Configuramos el layout y el botón de "Play"
def frame_args(duration):
    return {
        "frame": {"duration": duration},
        "mode": "immediate",
        "fromcurrent": True,
        "transition": {"duration": duration, "easing": "linear"},
    }

fig.update_layout(
    title=f"Descenso de Gradiente para {'Minimización' if min_or_max == 'min' else 'Maximización'}",
    scene=dict(
        xaxis_title='Eje X',
        yaxis_title='Eje Y',
        zaxis_title='f(x, y)'
    ),
    width=800,
    height=700,
    updatemenus=[{
        'type': 'buttons',
        'buttons': [
            {'label': 'Play',
             'method': 'animate',
             'args': [None, frame_args(50)]},
            {'label': 'Pause',
             'method': 'animate',
             'args': [[None], frame_args(0)]}
        ]
    }]
)

fig.show()

Punto inicial: (0.00, 0.00)
Punto final de la trayectoria: (-2.05, -1.60)
El valor de la integral de línea es: 1.6575
